# 01_Data_Preparation.ipynb

Build the initial `labeled_pd.parquet` from Spark FitnessBrowser tables.
This notebook produces the master table: one row per protein, with:
- Binary essentiality labels for every stressor (1 = essential, fitness < -2)
- Continuous fitness scores `{stressor}_fit` (NaN where no experiment was run)

The continuous fitness columns enable regression-based training in NB05, which
avoids the class-imbalance problem that degrades binary classification for metals.
They also distinguish true-negative proteins (fitness ≈ 0, experiment run) from
missing data (NaN, no experiment run for that organism/stressor pair).

### Imports & Spark session

In [1]:
import sys, json, logging, time, re
from pathlib import Path
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, types as T
from pyspark.sql.functions import regexp_replace, upper, trim

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

spark = get_spark_session()

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

### Stressor definitions

In [2]:
STRESSOR_PATTERNS = {
    # ── Metals ──────────────────────────────────────────────────
    'Zn':    ['zinc', 'zncl2', 'znso4', 'zinc chloride', 'zinc sulfate'],
    'Cu':    ['copper', 'cuso4', 'cucl2', 'copper sulfate', 'copper chloride'],
    'Cd':    ['cadmium', 'cdcl2', 'cdso4', 'cadmium chloride'],
    'Co':    ['cobalt', 'cocl2', 'coso4', 'cobalt chloride'],
    'Ni':    ['nickel', 'nicl2', 'niso4', 'nickel chloride', 'nickel sulfate'],
    'Cr':    ['chromium', 'chromate', 'k2cro4', 'dichromate', 'potassium chromate'],
    'As':    ['arsenate', 'arsenite', 'sodium arsenite', 'sodium arsenate', 'nah2aso4'],
    'Hg':    ['mercury', 'hgcl2', 'mercury chloride', 'mercuric chloride'],
    'Pb':    ['lead', 'pbno3', 'lead nitrate', 'pbcl2'],
    'Mn':    ['manganese', 'mncl2', 'mnso4', 'manganese chloride'],
    'Fe':    ['iron', 'feso4', 'fecl3', 'iron limitation', 'iron depletion', 'dipyridyl', 'bipyridyl'],
    'Se':    ['selenium', 'selenite', 'selenate', 'sodium selenite'],
    'Ag':    ['silver', 'agno3', 'silver nitrate'],
    'Al':    ['aluminium', 'aluminum', 'alcl3', 'aluminum chloride'],

    # ── Antibiotics ─────────────────────────────────────────────
    'Ampicillin':       ['ampicillin'],
    'Kanamycin':        ['kanamycin'],
    'Gentamicin':       ['gentamicin'],
    'Rifampicin':       ['rifampicin', 'rifampin'],
    'Chloramphenicol':  ['chloramphenicol'],
    'Tetracycline':     ['tetracycline'],
    'Phosphomycin':     ['phosphomycin', 'fosfomycin'],
    'Ceftazidime':      ['ceftazidime'],
    'Polymyxin':        ['polymyxin', 'colistin'],
    'Ramoplanin':       ['ramoplanin'],
    'Vancomycin':       ['vancomycin'],
    'Erythromycin':     ['erythromycin'],
    'Ciprofloxacin':    ['ciprofloxacin', 'cipro'],
    'Spectinomycin':    ['spectinomycin'],
    'Streptomycin':     ['streptomycin'],
    'Carbenicillin':    ['carbenicillin'],
    'Penicillin':       ['penicillin'],
    'Trimethoprim':     ['trimethoprim'],
    'Novobiocin':       ['novobiocin'],
    'Bacitracin':       ['bacitracin'],
    'Linezolid':        ['linezolid'],

    # ── Oxidative / Redox ──────────────────────────────────────
    'H2O2':     ['hydrogen peroxide', 'h2o2', 'peroxide'],
    'Paraquat': ['paraquat', 'methyl viologen'],
    'Menadione':['menadione', 'vitamin k3'],
    'Diamide':  ['diamide'],
    'Nitric oxide': ['nitric oxide', 'no', 'spermine noate', 'deta no'],

    # ── pH ─────────────────────────────────────────────────────
    'Acid':     ['acid', 'acidic', 'low ph', 'ph 4', 'ph 4.5', 'ph 5', 'hcl'],
    'Alkaline': ['alkaline', 'high ph', 'ph 9', 'ph 10', 'naoh', 'koh'],

    # ── Osmotic / Salts ─────────────────────────────────────────
    'NaCl':     ['sodium chloride', 'nacl', 'salt', 'salinity'],
    'KCl':      ['potassium chloride', 'kcl'],
    'Sucrose':  ['sucrose', 'osmotic'],

    # ── Temperature ────────────────────────────────────────────
    'Heat':     ['heat', 'high temperature', '42c', '45c', '37c'],
    'Cold':     ['cold', 'low temperature', '4c', '10c', '15c', '16c'],

    # ── Detergents / Membrane stress ───────────────────────────
    'SDS':          ['sds', 'sodium dodecyl sulfate', 'sodium dodecyl sulphate'],
    'EDTA':         ['edta', 'ethylenediaminetetraacetic acid'],
    'Ethanol':      ['ethanol', 'etoh'],
    'Bile salts':   ['bile', 'bile salts', 'cholate', 'deoxycholate'],

    # ── Nutrient limitation ────────────────────────────────────
    'Nitrogen limitation': ['nitrogen limit', 'n limitation', 'low nitrogen', 'ammonium limit'],
    'Phosphate limitation':['phosphate limit', 'p limitation', 'low phosphate'],
    'Iron limitation':     ['iron limit', 'fe limitation', 'low iron', 'dipyridyl'],
    'Carbon limitation':   ['carbon limit', 'c limitation', 'low carbon', 'minimal medium'],
    'Sulfur limitation':   ['sulfur limit', 'sulphur limit', 'low sulfate', 'low sulphur'],

    # ── Other ──────────────────────────────────────────────────
    'UV':       ['uv', 'ultraviolet', 'uv irradiation'],
    'Desiccation': ['desiccation', 'drying', 'dry'],
    'Anaerobic':   ['anaerobic', 'anoxic', 'low oxygen', 'nitrogen atmosphere'],
    'Biofilm':     ['biofilm'],
}

FITNESS_THRESHOLD = -2.0

### Extract per‑stressor labels from Spark

In [3]:
exp = spark.table("kescience_fitnessbrowser.experiment")
gf  = spark.table("kescience_fitnessbrowser.genefitness")

all_labels = None
for stressor, patterns in STRESSOR_PATTERNS.items():
    pattern = r'(?i)' + '|'.join(patterns)
    stressor_exps = exp.filter(F.col("expDesc").rlike(pattern)).select("expName")
    if stressor_exps.count() == 0:
        log.warning(f"No experiments for {stressor}. Skipping.")
        continue
    gene_fit = gf.join(stressor_exps, "expName") \
                 .withColumn("fit_num", F.col("fit").cast("float")) \
                 .groupBy("locusId", "orgId") \
                 .agg(F.min("fit_num").alias("fitness_min"))
    label = gene_fit \
        .withColumn(stressor,
                    F.when(F.col("fitness_min") < FITNESS_THRESHOLD, 1).otherwise(0)) \
        .withColumnRenamed("fitness_min", f"{stressor}_fit") \
        .select("locusId", "orgId", stressor, f"{stressor}_fit")
    if all_labels is None:
        all_labels = label
    else:
        all_labels = all_labels.join(label, ["locusId", "orgId"], "outer")

if all_labels is None:
    raise ValueError("No stressors matched.")

# Binary columns: fill missing with 0 (no experiment → treated as not essential)
# Continuous _fit columns: leave NaN for missing (no experiment → no label)
binary_cols = [c for c in all_labels.columns
               if c not in ("locusId", "orgId") and not c.endswith("_fit")]
all_labels = all_labels.fillna({c: 0 for c in binary_cols})

log.info(f"Spark labels: {all_labels.count()} genes × {len(binary_cols)} stressors")


### KEGG mapping to proteins

In [5]:
# Tables for mapping gene → protein
besthit = spark.table("kescience_fitnessbrowser.besthitkegg").select("locusId","orgId","keggOrg","keggId")
keggmember = spark.table("kescience_fitnessbrowser.keggmember").select("keggOrg","keggId","kgroup")
gene_kgroup = besthit.join(keggmember, ["keggOrg","keggId"]) \
                     .join(all_labels, ["locusId","orgId"])

# Get unique KEGG groups and their labels
stress_cols = [c for c in all_labels.columns if c not in ("locusId","orgId")]
ko_labels = gene_kgroup.groupBy('kgroup').agg(*(F.max(c).alias(c) for c in stress_cols))

# Clean KEGG IDs
def clean_kegg(col):
    return upper(trim(regexp_replace(col, "^ko:", "")))

kegg_orth = spark.table("enigma_genome_depot_enigma.browser_kegg_ortholog") \
                  .withColumn("kegg_id_clean", clean_kegg("kegg_id"))

# ko_labels is already a DataFrame – just add the cleaned column
ko_spark = ko_labels.withColumn("kgroup_clean", clean_kegg("kgroup"))

ko_with_id = kegg_orth.join(ko_spark,
                            kegg_orth.kegg_id_clean == ko_spark.kgroup_clean,
                            "inner") \
                     .select(kegg_orth.id.alias("kegg_ortholog_id"), *stress_cols)
matched = ko_with_id.count()
if matched == 0:
    raise ValueError("No KEGG orthologs matched.")

prot_labels = spark.table("enigma_genome_depot_enigma.browser_protein_kegg_orthologs") \
                    .join(ko_with_id, "kegg_ortholog_id") \
                    .select("protein_id", *stress_cols)
log.info(f"Proteins with labels: {prot_labels.count()}")

### Add sequences and organism names

In [8]:
protein = spark.table("enigma_genome_depot_enigma.browser_protein")
gene = spark.table("enigma_genome_depot_enigma.browser_gene")
genome = spark.table("enigma_genome_depot_enigma.browser_genome")
strain = spark.table("enigma_genome_depot_enigma.browser_strain")

labeled_with_seq = prot_labels \
    .join(protein, prot_labels.protein_id == protein.id, "inner") \
    .join(gene, protein.id == gene.protein_id, "inner") \
    .join(genome, gene.genome_id == genome.id, "inner") \
    .join(strain, genome.strain_id == strain.id, "inner") \
    .select(prot_labels.protein_id.alias("protein_id"),
            protein.sequence,
            strain.full_name.alias("organism"),
            *[prot_labels[c] for c in stress_cols])

# Write to a temporary Parquet file to avoid the 1 GB driver limit
temp_parquet = str(DATA_DIR / "labeled_temp.parquet")
labeled_with_seq.write.mode("overwrite").parquet(temp_parquet)

# Read back with pandas
labeled_pd = pd.read_parquet(temp_parquet)
log.info(f"Final proteins with sequences: {len(labeled_pd)}")

# Set protein_id as index
labeled_pd = labeled_pd.set_index('protein_id')

Error: 

### Determine active stressors and save

In [ ]:
MIN_POSITIVES = 30
stressor_counts = labeled_pd[stress_cols].sum()
active_stressors = stressor_counts[stressor_counts >= MIN_POSITIVES].index.tolist()
log.info(f"Active stressors: {len(active_stressors)} (≥{MIN_POSITIVES} positives)")

# Keep binary labels + continuous fitness scores for active stressors
fit_cols = [f"{s}_fit" for s in active_stressors if f"{s}_fit" in labeled_pd.columns]
keep_cols = ['sequence', 'organism'] + active_stressors + fit_cols
labeled_pd = labeled_pd[[c for c in keep_cols if c in labeled_pd.columns]]

labeled_pd.to_parquet(DATA_DIR / 'labeled_pd.parquet')
with open(DATA_DIR / 'active_stressors.json', 'w') as f:
    json.dump(active_stressors, f)

log.info(f"Saved labeled_pd.parquet: {len(labeled_pd)} proteins, "
         f"{len(active_stressors)} stressors, {len(fit_cols)} continuous fitness columns.")
